### **[Find the Duplicate Number (LeetCode 287)](https://leetcode.com/problems/find-the-duplicate-number/description/)**

This problem is a classic, but the constraints make it a deceptively complex puzzle.

First, let's address the follow-up question: *How can we prove that at least one duplicate number must exist?* This relies on a mathematical concept called the **Pigeonhole Principle**. Imagine you have 5 pigeonholes (the numbers 1 through 5) and 6 pigeons (the elements in the array). If every pigeon must go into a hole, at least one hole is guaranteed to contain more than one pigeon. Because we have $n + 1$ numbers squeezed into a range of $n$ possible values, a duplicate is mathematically inevitable.

Our goal is to find that exact "pigeonhole" that has multiple "pigeons" in it.

---

### **Constraints Analysis**

This problem is defined entirely by its notoriously strict constraints:

* `1 <= n <= 10^5`: The array can contain up to 100,000 elements.
* **Time Limitation:** An $O(n^2)$ approach will cause a Time Limit Exceeded (TLE) error. We need an $O(n \log n)$ or strictly $O(n)$ solution.
* **Space & Modification Limitation:** We **must not** modify the array (which rules out sorting it in-place), and we **must use $O(1)$ extra space** (which rules out creating a new sorted array or using a Hash Set to track seen numbers).

---

### **Approach Selection**

* **Pattern:** Linked List Cycle Detection (Fast and Slow Pointers).
* **Data Structure:** None (Conceptualizing the Array as a Linked List).
* **Reasoning:** Since we cannot use extra memory and cannot sort, we have to look at the relationship between the *indices* and the *values*. Because all values are strictly between $1$ and $n$, every value in the array perfectly points to a valid index within the array. If we treat the array as a graph where `index` points to `value`, multiple indices pointing to the same value (the duplicate) will create a cycle. We can use **Floyd’s Tortoise and Hare Algorithm** to find the entrance to that cycle.

---

### **Step-by-Step Implementation**

#### **1. Brute Force Approach (Compare All Pairs)**

The most fundamental way to find a duplicate is to take the first number and check the rest of the array to see if it appears again. If not, move to the second number and repeat.

```python
class Solution:
    def findDuplicate(self, nums: list[int]) -> int:
        n = len(nums)
        for i in range(n):
            for j in range(i + 1, n):
                if nums[i] == nums[j]:
                    return nums[i]

```

* **Time Complexity:** $O(n^2)$. We compare nearly every possible pair. Fails the $10^5$ constraint.
* **Space Complexity:** $O(1)$.
* **Interview Reasoning:** Mention this briefly to establish a baseline, but immediately acknowledge it is too slow.

#### **2. Better Approach (Hash Set Tracker)**

If we were allowed to use extra space, we could just write down every number we see as we iterate through the array. The moment we try to write down a number we've already recorded, we've found our duplicate.

```python
class Solution:
    def findDuplicate(self, nums: list[int]) -> int:
        seen = set()
        for num in nums:
            if num in seen:
                return num
            seen.add(num)

```

* **Time Complexity:** $O(n)$. We pass through the array once.
* **Space Complexity:** $O(n)$. We create a Hash Set that scales with the input size.
* **Interview Reasoning:** This is exactly what an interviewer expects you to suggest first to prove you know how to solve duplicate problems in $O(n)$ time. You would then say, *"However, this violates the $O(1)$ space constraint, meaning I need to find a way to track traversal without allocating memory."* (Note: Mentioning sorting here is also good, but quickly dismiss it as it violates the "do not modify" constraint).

#### **3. Optimal Approach (Floyd's Tortoise and Hare)**

We treat the array like a linked list. The current index is the "Node", and the value at that index is the "Next Pointer".
For `nums = [1, 3, 4, 2, 2]`:

* Index 0 points to 1
* Index 1 points to 3
* Index 3 points to 2
* Index 2 points to 4
* Index 4 points to 2 (Cycle created here!)

We use a `slow` pointer (moves 1 step) and a `fast` pointer (moves 2 steps). Eventually, they will crash into each other inside the cycle. Once they meet, we leave one pointer at the intersection, move the other back to the start (index 0), and move *both* at 1 step at a time. The exact node where they collide again is mathematically guaranteed to be the entrance to the cycle (the duplicate number).

```python
class Solution:
    def findDuplicate(self, nums: list[int]) -> int:
        # Phase 1: Find the intersection point in the cycle
        slow = nums[0]
        fast = nums[nums[0]]
        
        while slow != fast:
            slow = nums[slow]           # Moves 1 step
            fast = nums[nums[fast]]     # Moves 2 steps
            
        # Phase 2: Find the entrance to the cycle
        slow = 0 # Reset slow pointer to the start
        
        while slow != fast:
            slow = nums[slow]           # Both now move 1 step
            fast = nums[fast]
            
        return slow # The meeting point is the duplicate

```

* **Time Complexity:** $O(n)$. Both phases scale linearly with the size of the array.
* **Space Complexity:** $O(1)$. We strictly use two pointer variables, perfectly satisfying all constraints.
* **Interview Reasoning:** This is the absolute optimal solution and a classic "trick" to be aware of. Recognizing that values constrained to valid bounds can represent graph edges is a high-level deductive skill.